Pre-trained VGG19 / ResNet18 / ResNet34 inference on dummy data of shape (B, 3, 224,224)



In [1]:
#Pre-trained VGG19 / ResNet18 / ResNet34 inference on dummy data of shape (B, 3, 224,224)

import torch
import torch.nn.functional as F
from torchvision.models import (
    vgg19, VGG19_Weights,
    resnet18, ResNet18_Weights,
    resnet34, ResNet34_Weights
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
dummy_x = torch.randn(2, 3, 224, 224).to(device)  #batch of 2

vgg_weights = VGG19_Weights.DEFAULT
resnet18_weights = ResNet18_Weights.DEFAULT
resnet34_weights = ResNet34_Weights.DEFAULT

vgg_model = vgg19(weights=vgg_weights).to(device).eval()  #Don't put eval, if you wanna do training
res18_model = resnet18(weights=resnet18_weights).to(device).eval()
res34_model = resnet34(weights=resnet34_weights).to(device).eval()

Downloading: "https://download.pytorch.org/models/vgg19-dcbb9e9d.pth" to /root/.cache/torch/hub/checkpoints/vgg19-dcbb9e9d.pth


100%|██████████| 548M/548M [00:07<00:00, 73.1MB/s]


Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to /root/.cache/torch/hub/checkpoints/resnet18-f37072fd.pth


100%|██████████| 44.7M/44.7M [00:00<00:00, 103MB/s]


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth


100%|██████████| 83.3M/83.3M [00:00<00:00, 91.1MB/s]


In [3]:
with torch.no_grad():
    vgg_logits = vgg_model(dummy_x)     #(2,1000)
    res18_logits = res18_model(dummy_x) #(2,1000)
    res34_logits = res34_model(dummy_x) #(2,1000)

vgg_probs = F.softmax(vgg_logits, dim=1)
res18_probs = F.softmax(res18_logits, dim=1)
res34_probs = F.softmax(res34_logits, dim=1)

print("VFF19 logits shape:", vgg_logits.shape)
print("VGG19 probs shape:", vgg_probs.shape)
print("ResNet18 logits shape:", res18_logits.shape)
print("ResNet18 probs shape:", res18_probs.shape)
print("ResNet34 logits shape:", res34_logits.shape)
print("ResNet34 probs shape:", res34_probs.shape)

def print_top5(probs, weights, model_name):
    categories = weights.meta["categories"]
    for i in range(probs.size(0)):
        top5_prob, top5_idx = torch.topk(probs[i], k=5)
        print(f"{model_name} sample {i}:")
        for rank, (p, idx) in enumerate(zip(top5_prob, top5_idx), start =1):
            print(f"Top{rank}: Class ID={idx.item():4d},"
                  f"Label={categories[idx.item()]}, "
                  f"Prob={p.item():.6f}")
print_top5(vgg_probs, vgg_weights, "VGG19")
print_top5(res18_probs, resnet18_weights, "ResNet18")
print_top5(res34_probs, resnet34_weights, "ResNet34")


VFF19 logits shape: torch.Size([2, 1000])
VGG19 probs shape: torch.Size([2, 1000])
ResNet18 logits shape: torch.Size([2, 1000])
ResNet18 probs shape: torch.Size([2, 1000])
ResNet34 logits shape: torch.Size([2, 1000])
ResNet34 probs shape: torch.Size([2, 1000])
VGG19 sample 0:
Top1: Class ID= 794,Label=shower curtain, Prob=0.073770
Top2: Class ID= 885,Label=velvet, Prob=0.069702
Top3: Class ID= 556,Label=fire screen, Prob=0.038602
Top4: Class ID= 619,Label=lampshade, Prob=0.031124
Top5: Class ID= 904,Label=window screen, Prob=0.028197
VGG19 sample 1:
Top1: Class ID= 909,Label=wok, Prob=0.041335
Top2: Class ID= 885,Label=velvet, Prob=0.039312
Top3: Class ID= 904,Label=window screen, Prob=0.028254
Top4: Class ID= 794,Label=shower curtain, Prob=0.028194
Top5: Class ID= 632,Label=loudspeaker, Prob=0.025224
ResNet18 sample 0:
Top1: Class ID= 107,Label=jellyfish, Prob=0.125641
Top2: Class ID= 611,Label=jigsaw puzzle, Prob=0.053394
Top3: Class ID= 845,Label=syringe, Prob=0.031327
Top4: Class I

In [4]:
print(vgg_model)
#have to modify the classifier head if input dim of image changes

VGG(
  (features): Sequential(
    (0): Conv2d(3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padd

In [5]:
#modify VGG19 head for custom classes

import torch
import torch.nn as nn
from torchvision.models import vgg19, VGG19_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 5

model_vgg = vgg19(weights=VGG19_Weights.DEFAULT)

#replace final classification Layer
in_features = model_vgg.classifier[6].in_features  #4096
model_vgg.classifier[6] = nn.Linear(in_features, num_classes) #modification to layer 6 of classifier
#random 4096*5 matrix

model_vgg = model_vgg.to(device)
model_vgg.train()

dummy_x = torch.randn(4, 3, 224, 224).to(device)
logits = model_vgg(dummy_x)
print("Modified VGG19 output shape", logits.shape)   #(4,5)

Modified VGG19 output shape torch.Size([4, 5])


In [6]:
#modify ResNet18, ResNet34 head for custom classes



import torch
import torch.nn as nn
from torchvision.models import (
    resnet18, ResNet18_Weights,
    resnet34, ResNet34_Weights
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
num_classes = 5

# ResNet18
model_res18 = resnet18(weights=ResNet18_Weights.DEFAULT)
in_features_res18 = model_res18.fc.in_features
model_res18.fc = nn.Linear(in_features_res18, num_classes)
model_res18 = model_res18.to(device)

# ResNet34
model_res34 = resnet34(weights=ResNet34_Weights.DEFAULT)
in_features_res34 = model_res34.fc.in_features
model_res34.fc = nn.Linear(in_features_res34, num_classes)
model_res34 = model_res34.to(device)

dummy_x = torch.randn(4, 3, 224, 224).to(device)

out18 = model_res18(dummy_x)
out34 = model_res34(dummy_x)

print("Modified ResNet18 output shape:", out18.shape)  # (4, 5)
print("Modified ResNet34 output shape:", out34.shape)  # (4, 5)

Modified ResNet18 output shape: torch.Size([4, 5])
Modified ResNet34 output shape: torch.Size([4, 5])


In [7]:
#extract embedding from VGG19 (feature extraction)
# used in IMAGE CAPTIONING

#no need of classifier, get the model until avgpool, just before classifier

import torch
import torch.nn as nn
from torchvision.models import vgg19, VGG19_Weights

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class VGG19Embedding(nn.Module):
    def __init__(self, weights=VGG19_Weights.DEFAULT):
        super().__init__()
        base_model = vgg19(weights=weights)

        self.features = base_model.features
        self.avgpool = base_model.avgpool
        # We do NOT use the classifier

    def forward(self, x):
        x = self.features(x)     # (B, 512, 7, 7) for 224x224 input
        x = self.avgpool(x)      # (B, 512, 7, 7)
        x = torch.flatten(x, 1)  # (B, 25088)  #--embedding
        return x

model_vgg_embed = VGG19Embedding().to(device).eval()

dummy_x = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    embeddings = model_vgg_embed(dummy_x)

print("VGG19 embedding shape:", embeddings.shape)  # (2, 25088)

VGG19 embedding shape: torch.Size([2, 25088])


In [8]:
#feature extractor (extract embeddings) from ResNet

import torch
import torch.nn as nn
from torchvision.models import (
    resnet18, ResNet18_Weights,
    resnet34, ResNet34_Weights
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class ResNetEmbedding(nn.Module):
    def __init__(self, arch="resnet18"):
        super().__init__()

        if arch == "resnet18":
            base_model = resnet18(weights=ResNet18_Weights.DEFAULT)
        elif arch == "resnet34":
            base_model = resnet34(weights=ResNet34_Weights.DEFAULT)
        else:
            raise ValueError("arch must be 'resnet18' or 'resnet34'")

        self.stem = nn.Sequential(
            base_model.conv1,
            base_model.bn1,
            base_model.relu,
            base_model.maxpool
        )

        self.layer1 = base_model.layer1
        self.layer2 = base_model.layer2
        self.layer3 = base_model.layer3
        self.layer4 = base_model.layer4
        self.avgpool = base_model.avgpool
        #self.fc is intentially removed

    def forward(self, x):
        x = self.stem(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.avgpool(x)      # (B, 512, 1, 1)
        x = torch.flatten(x, 1)  # (B, 512)
        return x

model_res18_embed = ResNetEmbedding(arch="resnet18").to(device).eval()
model_res34_embed = ResNetEmbedding(arch="resnet34").to(device).eval()

dummy_x = torch.randn(2, 3, 224, 224).to(device)

with torch.no_grad():
    emb18 = model_res18_embed(dummy_x)
    emb34 = model_res34_embed(dummy_x)

print("ResNet18 embedding shape:", emb18.shape)  # (2, 512)
print("ResNet34 embedding shape:", emb34.shape)  # (2, 512)

ResNet18 embedding shape: torch.Size([2, 512])
ResNet34 embedding shape: torch.Size([2, 512])
